# SNOOSE
### This notebook queries JPL Horizons for an object's ephemeris over a specified UTC time range and visualizes:
#### - Sky positions (RA/Dec) as magenta cross markers
#### - 3σ positional uncertainties as red ellipses
#### The visualization is rendered using Aladin Lite and embedded directly in the notebook via a base64 HTML IFrame.

### User Inputs
#### Object Identifier ID:
##### - Examples: "3I", "P/2020 MK4", "73P-AB"
##### - For fragmented objects - specify which piece :)
#### Observatory Location:
##### - MPC Observatory Code
##### - Examples: Rubin Observatory = "X05", Apache Point Observatory = "705"
#### Epoch:
##### - In UTC
#####   "start": "YYYY-MM-DD HH:MM",
#####    "stop":  "YYYY-MM-DD HH:MM",
#####    "step":  "1m" | "5m" | "1h" | etc.

In [1]:
from astroquery.jplhorizons import Horizons
from IPython.display import IFrame
import pandas as pd, base64, json

# get ephemeris with 3σ columns
h = Horizons(
    id="3I",   # technically 3I/ATLAS
    location="705",   # obs code
    epochs={"start":"2025-11-13 06:20", #ITS IN UTC
            "stop":"2025-11-13 06:30",
            "step":"1m"}  
    # hourly time + stepsize
)
df = (
    h.ephemerides().to_pandas().rename(columns={
        "RA":"RA_deg","DEC":"DEC_deg",
        "RA_3sigma":"RA3_asec","DEC_3sigma":"DEC3_asec"
    })
)

# center so aladin knows where to point and data array
ra0, dec0 = df.RA_deg.mean(), df.DEC_deg.mean()
records = df[["RA_deg","DEC_deg","datetime_str","RA3_asec","DEC3_asec"]].to_dict("records")
js_array = json.dumps(records)

# makes HTML page (why are markers so tough to display!!!)
html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Aladin Lite Unc</title>
  <script src="https://aladin.cds.unistra.fr/AladinLite/api/v3/latest/aladin.js"></script>
  <style>body,html {{margin:0;padding:0;height:100%;}}</style>
</head><body>
  <div id="aladin-lite-div" style="width:100%; height:100vh;"></div>
  <script>
  A.init.then(() => {{
    const aladin = A.aladin('#aladin-lite-div', {{
      survey:   'P/DSS2/color',
      target:   '{ra0:.6f} {dec0:.6f}',
      fov:      0.5,
      cooFrame: 'ICRS'
    }});

    // cross markers
    const trackLayer = A.catalog({{ name:'Track', shape:'cross', color:'magenta', sourceSize:10 }});
    aladin.addCatalog(trackLayer);

    // overlay for red ellipses
    const overlay = A.graphicOverlay({{ color:'red', lineWidth:2 }});
    aladin.addOverlay(overlay);

    const data = {js_array};

    data.forEach(d => {{
      // magenta cross
      trackLayer.addSources([
        A.marker(d.RA_deg, d.DEC_deg, {{
          popupTitle: d.datetime_str,
          popupDesc:  'σ_RA='+(d.RA3_asec/3600).toFixed(4)+'°, σ_Dec='+(d.DEC3_asec/3600).toFixed(4)+'°'
        }})
      ]); 

      // red uncertainty ellipse
      const raSig  = d.RA3_asec/3600;
      const decSig = d.DEC3_asec/3600;
      const ell = A.ellipse(d.RA_deg, d.DEC_deg, raSig, decSig, 0, {{}});
      overlay.add(ell);
    }});
   // rotate and flip
    const controls = document.createElement('div');
    controls.style.position = 'absolute';
    controls.style.top = '10px';
    controls.style.right = '10px';
    controls.style.background = 'rgba(255,255,255,0.8)';
    controls.style.padding = '8px';
    controls.innerHTML = `
      <label>Rotate: <input type="range" id="rot-slider" min="0" max="360" value="0"></label><br>
      <label><input type="checkbox" id="flip-toggle"> Flip Horizontally</label>
    `;
    document.body.appendChild(controls);

    const rotSlider = document.getElementById('rot-slider');
    const flipToggle = document.getElementById('flip-toggle');

    rotSlider.addEventListener('input', () => {{
      const angle = parseInt(rotSlider.value);
      aladin.setRotation(angle);
    }});
    flipToggle.addEventListener('change', () => {{
      const flipped = flipToggle.checked;
      document.getElementById('aladin-lite-div').style.transform = flipped ? 'scaleX(-1)' : '';
    }});
    // 

  }});
  </script>
</body></html>
"""

# embedding via base 64 and Iframe cause it helps (somehow)
b64 = base64.b64encode(html.encode('utf-8')).decode('ascii')
uri = 'data:text/html;base64,' + b64
IFrame(src=uri, width=800, height=600)


##### 3I, asteroid, fragment 
##### write up how this works, what stuff you need to provide, providing 
##### check if uncertainty ellipse works 

In [8]:
from astroquery.jplhorizons import Horizons
from IPython.display import IFrame
import pandas as pd
import base64, json

# USER INPUTS

TARGET_ID = "90000395"
OBS_CODE  = "705"
EPOCHS = {
    "start": "2025-11-13 06:20",  # UTC
    "stop":  "2025-11-13 06:30",  # UTC
    "step":  "1m",
}
QUANTITIES = "1,3,9,36,41"  



# QUERY HORIZONS

h = Horizons(id=TARGET_ID, location=OBS_CODE, epochs=EPOCHS)

try:
    eph = h.ephemerides(quantities=QUANTITIES)
except TypeError:
    eph = h.ephemerides()

raw = eph.to_pandas()



# PICK ANOMALY COLUMN

def pick_anomaly_column(columns):
    cols = list(columns)
    lower = {c: c.strip().lower() for c in cols}

    # Mean anomaly preferred
    if "M" in cols:
        return "M", "Mean anomaly"
    for c in cols:
        if "mean" in lower[c] and "anom" in lower[c]:
            return c, "Mean anomaly"

    # True anomaly fallback
    if "nu" in cols:
        return "nu", "True anomaly"
    for c in cols:
        if "true" in lower[c] and "anom" in lower[c]:
            return c, "True anomaly"

    return None, None


anom_col, anom_label = pick_anomaly_column(raw.columns)
if anom_col is None:
    anom_like = [c for c in raw.columns if ("anom" in c.lower()) or (c.strip().lower() in {"m", "nu"})]
    raise KeyError(
        "No mean/true anomaly column found in Horizons output.\n"
        f"Anomaly-like columns present: {anom_like}\n"
        "Debug with:\n"
        "  print(raw.columns.tolist())"
    )



# RENAME + PACK FOR JS

df = raw.rename(columns={
    "RA": "RA_deg",
    "DEC": "DEC_deg",
    "RA_3sigma": "RA3_asec",
    "DEC_3sigma": "DEC3_asec",
    anom_col: "anom_deg",
})

required = ["RA_deg", "DEC_deg", "RA3_asec", "DEC3_asec", "datetime_str", "anom_deg"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}\nAvailable columns: {df.columns.tolist()}")

ra0, dec0 = df["RA_deg"].mean(), df["DEC_deg"].mean()
records = df[["RA_deg", "DEC_deg", "datetime_str", "RA3_asec", "DEC3_asec", "anom_deg"]].to_dict("records")
js_array = json.dumps(records)



html = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Aladin Lite Track + Uncertainty + Anomaly</title>
  <script src="https://aladin.cds.unistra.fr/AladinLite/api/v3/latest/aladin.js"></script>
  <style>
    html, body {{
      margin: 0;
      padding: 0;
      height: 100%;
      font-family: Arial, sans-serif;
    }}

    /* Wrapper that holds everything */
    #wrap {{
      position: relative;
      width: 100%;
      height: 100vh;
      overflow: hidden;
    }}

    /* Aladin container: MUST remain empty (Aladin owns its DOM) */
    #aladin-lite-div {{
      width: 100%;
      height: 100%;
    }}

    /* Overlays */
    #controls {{
      position: absolute;
      top: 10px;
      right: 320px;
      background: rgba(255,255,255,0.85);
      padding: 8px 10px;
      border: 1px solid #ccc;
      z-index: 9999;
    }}

    #sidepanel {{
      position: absolute;
      top: 10px;
      right: 10px;
      width: 300px;
      background: rgba(255,255,255,0.92);
      border: 1px solid #ccc;
      padding: 10px 12px;
      box-sizing: border-box;
      z-index: 9999;
    }}

    #sidepanel h3 {{
      margin: 0 0 8px 0;
      font-size: 16px;
    }}

    .kv {{
      margin: 6px 0;
      font-size: 13px;
      line-height: 1.25;
    }}

    .kv b {{
      display: inline-block;
      width: 105px;
    }}
  </style>
</head>

<body>
  <div id="wrap">
    <div id="aladin-lite-div"></div>

    <div id="controls">
      <label>Rotate:
        <input type="range" id="rot-slider" min="0" max="360" value="0">
      </label><br>
      <label>
        <input type="checkbox" id="flip-toggle"> Flip Horizontally
      </label>
    </div>

    <div id="sidepanel">
      <h3>Ephemeris Point</h3>
      <div class="kv"><b>Time:</b> <span id="sp-time">—</span></div>
      <div class="kv"><b>RA, Dec:</b> <span id="sp-radec">—</span></div>
      <div class="kv"><b>3σ RA:</b> <span id="sp-ra3">—</span></div>
      <div class="kv"><b>3σ Dec:</b> <span id="sp-dec3">—</span></div>
      <div class="kv"><b>{anom_label}:</b> <span id="sp-anom">—</span></div>
      <div class="kv" style="color:#666;font-size:12px;margin-top:8px;">
        Click a marker to update.
      </div>
    </div>
  </div>

<script>
A.init.then(() => {{
  const aladin = A.aladin('#aladin-lite-div', {{
    survey:   'P/DSS2/color',
    target:   '{ra0:.6f} {dec0:.6f}',
    fov:      0.5,
    cooFrame: 'ICRS'
  }});

  const trackLayer = A.catalog({{ name:'Track', shape:'cross', color:'magenta', sourceSize:10 }});
  aladin.addCatalog(trackLayer);

  const overlay = A.graphicOverlay({{ color:'red', lineWidth:2 }});
  aladin.addOverlay(overlay);

  const data = {js_array};

  function updatePanel(d) {{
    document.getElementById('sp-time').textContent  = d.datetime_str;
    document.getElementById('sp-radec').textContent = d.RA_deg.toFixed(6) + ', ' + d.DEC_deg.toFixed(6);
    document.getElementById('sp-ra3').textContent   = d.RA3_asec.toFixed(2) + ' arcsec';
    document.getElementById('sp-dec3').textContent  = d.DEC3_asec.toFixed(2) + ' arcsec';
    document.getElementById('sp-anom').textContent  = d.anom_deg.toFixed(3) + ' deg';
  }}
  if (data.length > 0) updatePanel(data[0]);

  data.forEach(d => {{
    const m = A.marker(d.RA_deg, d.DEC_deg, {{
      popupTitle: d.datetime_str,
      popupDesc:  '{anom_label}: ' + d.anom_deg.toFixed(3) + '°'
                + '<br>3σ_RA: ' + d.RA3_asec.toFixed(2) + '″'
                + '<br>3σ_Dec: ' + d.DEC3_asec.toFixed(2) + '″'
    }});
    m.data = d;
    trackLayer.addSources([m]);

    const raSigDeg  = d.RA3_asec / 3600.0;
    const decSigDeg = d.DEC3_asec / 3600.0;
    overlay.add(A.ellipse(d.RA_deg, d.DEC_deg, raSigDeg, decSigDeg, 0, {{}}));
  }});

  try {{
    trackLayer.on('click', (source) => {{
      if (source && source.data) updatePanel(source.data);
    }});
  }} catch(e) {{}}


  // ROTATE + FLIP (CSS transform)

  const rotSlider  = document.getElementById('rot-slider');
  const flipToggle = document.getElementById('flip-toggle');

  let rotDeg = 0;
  let flipX  = false;
  let renderEl = null;

  function findRenderEl() {{
    const root = document.getElementById('aladin-lite-div');
    // Pick the element that visually rotates the sky (canvas wrapper)
    const canvas = root.querySelector('canvas');
    if (!canvas) return null;
    return canvas.parentElement || canvas;
  }}

  function applyTransform() {{
    if (!renderEl) return;
    const sx = flipX ? -1 : 1;
    renderEl.style.transformOrigin = 'center center';
    renderEl.style.transform = `scaleX(${{sx}}) rotate(${{rotDeg}}deg)`;
  }}

  function ensureRenderElAndApply() {{
    renderEl = findRenderEl();
    if (renderEl) {{
      applyTransform();
      return true;
    }}
    return false;
  }}

  // Try immediately, then observe DOM changes until canvas appears
  if (!ensureRenderElAndApply()) {{
    const obs = new MutationObserver(() => {{
      if (ensureRenderElAndApply()) obs.disconnect();
    }});
    obs.observe(document.getElementById('aladin-lite-div'), {{ childList: true, subtree: true }});
  }}

  rotSlider.addEventListener('input', () => {{
    rotDeg = parseFloat(rotSlider.value) || 0;
    applyTransform();
  }});

  flipToggle.addEventListener('change', () => {{
    flipX = !!flipToggle.checked;
    applyTransform();
  }});
}});
</script>

</body>
</html>
"""


b64 = base64.b64encode(html.encode("utf-8")).decode("ascii")
uri = "data:text/html;base64," + b64
IFrame(src=uri, width=1100, height=650)